## 1. Setup

In [ ]:
import os
import sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q transformers arabert emoji tashaphyne nltk

Results, predictions and checkpoints are written straight to Google Drive, so a disconnected session loses nothing.

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/Arabic-sarcasm")
DRIVE = Path("/content/drive/MyDrive/sarcasm")

for name in ["results", "predictions", "models"]:
    local, remote = ROOT / name, DRIVE / name
    if not remote.exists():                       # first time: copy what the repo already has
        if local.exists():
            shutil.copytree(local, remote)
        else:
            remote.mkdir(parents=True)
    if local.exists() and not local.is_symlink():
        shutil.rmtree(local)
    if not local.exists():
        local.symlink_to(remote)

## 2. Data

In [ ]:
import sys
sys.path.insert(0, "/content/Arabic-sarcasm")

import warnings

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from transformers import logging as hf_logging

from src.data import get_data
from src.evaluate import RESULTS, log_result, metrics, plot_results, save_predictions, show
from src.fine_tune import DEVICE, PLMS, MultiTask, SingleTask, predict_proba, prepare_text, train_model
from src.train import best_threshold, count_params, set_seed

hf_logging.set_verbosity_error()      # hide the model loading report
hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message="Detected call of")   # harmless: fp16 skips the first optimizer step

MODEL = "marbert"
PLM = PLMS[MODEL]
MAX_LEN = PLM["max_len"]
SEEDS = [42, 43, 44]
LAMS = [0.3, 0.5, 1.0]

train, val, test, report = get_data()
print(report)

Texts are prepared once for all runs. Check 5 tweets before and after, and how many tweets are cut by `max_len`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PLM["hf_name"])

x_train = prepare_text(MODEL, train["tweet"])
x_val = prepare_text(MODEL, val["tweet"])
x_test = prepare_text(MODEL, test["tweet"])

y_train = train["sarcasm"].to_numpy()
s_train = train["sentiment_id"].to_numpy()
y_val = val["sarcasm"].to_numpy()
y_test = test["sarcasm"].to_numpy()

for before, after in zip(train["tweet"].head(5), x_train[:5]):
    print(before)
    print(after)
    print()

lengths = np.array([len(tokenizer(t)["input_ids"]) for t in x_train])
print("tweets longer than max_len:", (lengths > MAX_LEN).mean())

## 3. What one run does

In [ ]:
def make_model(task, lam):
    if task == "single":
        model = SingleTask(PLM["hf_name"])
    else:
        model = MultiTask(PLM["hf_name"], lam)
    return model.to(DEVICE)


def already_done(name, seed):
    if not RESULTS.exists() or RESULTS.stat().st_size == 0:
        return False
    table = pd.read_csv(RESULTS)
    same = (table["model"] == name) & (table["preprocessing"] == PLM["prep"]) & (table["seed"] == seed)
    return bool(same.any())

In [ ]:
def run_one(task, lam, seed):
    name = f"{MODEL}_{task}_lam{lam}"
    if already_done(name, seed):
        print("already done:", name, "seed", seed)
        return

    # 1. train (the model is left at its best epoch)
    set_seed(seed)
    model = make_model(task, lam)
    train_time = train_model(model, tokenizer, x_train, y_train, s_train, x_val, y_val, MAX_LEN)

    # 2. validation: choose the threshold
    val_proba = predict_proba(model, tokenizer, x_val, MAX_LEN)
    threshold = best_threshold(y_val, val_proba)
    val_scores = metrics(y_val, val_proba, threshold)

    # 3. test: use the validation threshold
    test_proba = predict_proba(model, tokenizer, x_test, MAX_LEN)
    test_scores = metrics(y_test, test_proba, threshold)
    save_predictions(name, seed, test["id"], y_test, test_proba, threshold)

    # 4. log
    row = {"model": name, "preprocessing": PLM["prep"], "seed": seed, "lam": lam,
           "threshold": threshold, "train_time": train_time, "trained_params": count_params(model)}
    row.update({"val_" + k: v for k, v in val_scores.items()})
    row.update({"test_" + k: v for k, v in test_scores.items()})
    log_result(**row)
    show(name, val_scores, threshold, train_time)

    # 5. checkpoint, first seed only
    if seed == SEEDS[0]:
        torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                    "hf_name": PLM["hf_name"], "task": task, "lam": lam, "prep": PLM["prep"],
                    "max_len": MAX_LEN, "threshold": threshold},
                   ROOT / "models" / f"{name}_seed{seed}.pt")

    torch.cuda.empty_cache()

In [ ]:
run_one("single", 0, SEEDS[0]) #42

In [ ]:
run_one("single", 0, SEEDS[1]) #43

In [ ]:
run_one("single", 0, SEEDS[2]) #44

In [ ]:
run_one("multi", LAMS[0], SEEDS[0]) #42


In [ ]:
run_one("multi", LAMS[1], SEEDS[0]) #42

In [ ]:
run_one("multi", LAMS[2], SEEDS[0]) #42

In [ ]:
run_one("multi", LAMS[0], SEEDS[1]) #43

In [ ]:
run_one("multi", LAMS[1], SEEDS[1]) #43

In [ ]:
run_one("multi", LAMS[2], SEEDS[1]) #43

In [ ]:
run_one("multi", LAMS[0], SEEDS[2]) #44

In [ ]:
run_one("multi", LAMS[1], SEEDS[2]) #44

In [ ]:
run_one("multi", LAMS[2], SEEDS[2]) #44

## 5. Results
Validation F1 of the sarcastic class, mean and std over the 3 seeds (`count` must be 3 for every row).

In [ ]:
results = pd.read_csv(RESULTS)
results = results[(results["preprocessing"] == PLM["prep"]) & results["model"].str.contains("_single_|_multi_")]

val_table = results.groupby("model")["val_f1"].agg(["mean", "std", "count"])
val_table

In [ ]:
plot_table = results.groupby("model")[["val_f1", "val_pr_auc"]].mean()
plot_table.columns = ["f1", "pr_auc"]
plot_results(plot_table)

λ is chosen from the validation table. Test is shown only for E1 and the chosen E2.

In [ ]:
multi = val_table[val_table.index.str.contains("_multi_")]
best_e2 = multi["mean"].idxmax()
print("chosen E2:", best_e2)

chosen = [f"{MODEL}_single_lam0", best_e2]
results[results["model"].isin(chosen)].groupby("model")["test_f1"].agg(["mean", "std"])